# RNNs: state across time and a vanishing-gradient illustration

Companion to HTB Academy Module 290, Section 20. This is a **scalar teaching recurrence**, not HTB's prescribed implementation or a trained text model. It shows what it means to reuse a cell and carry state forward.

Read hₜ=tanh(wₓxₜ+wₕhₜ₋₁+b) as **h sub t equals tanh of w sub x times x sub t, plus w sub h times h sub t-minus-one, plus b**. The current input and previous state jointly determine the next state.

In [ ]:
import math

def trace_states(sequence, input_weight=1.0, recurrent_weight=0.5, bias=0.0):
    """Apply the same illustrative cell weights at every ordered time step."""
    state = 0.0  # h_0: no earlier observations in this toy sequence.
    states = []
    for time_step, current_input in enumerate(sequence, start=1):
        pre_activation = input_weight * current_input + recurrent_weight * state + bias
        new_state = math.tanh(pre_activation)
        print(f't={time_step}: x={current_input}, previous h={state:.3f}, input to tanh={pre_activation:.3f}, new h={new_state:.3f}')
        states.append(new_state)
        state = new_state  # Carry this summary into the next time step.
    return states

forward_states = trace_states((1, 0, 0))
print('reverse order:')
reverse_states = trace_states((0, 0, 1))
assert 0.76 < forward_states[0] < 0.77
assert 0.17 < forward_states[-1] < 0.19
assert 0.76 < reverse_states[-1] < 0.77
assert forward_states[-1] != reverse_states[-1]

## Why backward signals can shrink

In backpropagation through time, local chain-rule factors multiply as the training signal passes backward across steps. If each factor were 0.5, the signal after n steps would be scaled by 0.5ⁿ, read **one half to the n-th power**. Real RNN derivatives vary; these powers isolate the mechanism, not an exact model calculation. Gates in LSTMs and GRUs can create better routes for long-range information and gradients.

In [ ]:
local_factor = 0.5
for steps_back in (1, 5, 10, 20):
    remaining = local_factor ** steps_back  # Product of equal toy chain-rule factors.
    print(f'{steps_back:2d} steps back: {remaining:.10f} of the signal remains')
assert local_factor ** 10 == 0.0009765625
assert local_factor ** 20 < 1e-6